In [1]:
!cp -r /kaggle/input/datasets/baohg153/prover9/Prover9-LADR-2026-4F /kaggle/working/

%cd /kaggle/working/Prover9-LADR-2026-4F
!make all STATIC=1
%cd /kaggle/working/

/kaggle/working/Prover9-LADR-2026-4F
cd ladr         && make lib
make[1]: Entering directory '/kaggle/working/Prover9-LADR-2026-4F/ladr'
make libladr.a
make[2]: Entering directory '/kaggle/working/Prover9-LADR-2026-4F/ladr'
ladr: build mode changed to release, rebuilding...
gcc -O2 -Wall -MMD -MP     -c -o order.o order.c
gcc -O2 -Wall -MMD -MP     -c -o clock.o clock.c
gcc -O2 -Wall -MMD -MP     -c -o nonport.o nonport.c
gcc -O2 -Wall -MMD -MP     -c -o fatal.o fatal.c
gcc -O2 -Wall -MMD -MP     -c -o ibuffer.o ibuffer.c
gcc -O2 -Wall -MMD -MP     -c -o memory.o memory.c
gcc -O2 -Wall -MMD -MP     -c -o hash.o hash.c
gcc -O2 -Wall -MMD -MP     -c -o string.o string.c
gcc -O2 -Wall -MMD -MP     -c -o strbuf.o strbuf.c
gcc -O2 -Wall -MMD -MP     -c -o glist.o glist.c
gcc -O2 -Wall -MMD -MP     -c -o options.o options.c
gcc -O2 -Wall -MMD -MP     -c -o symbols.o symbols.c
gcc -O2 -Wall -MMD -MP     -c -o avltree.o avltree.c
gcc -O2 -Wall -MMD -MP     -c -o term.o term.c
gcc -O2 -Wall -MM

In [2]:
!pip install pandas \
            seaborn \
            matplotlib \
            nltk \
            regex \
            torch \
            datasets \
            transformers \
            peft \
            nltk

In [3]:
import torch
import json
from datasets import Dataset,DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel
from torch.nn import CrossEntropyLoss

import random
import numpy as np

import nltk
from nltk.sem.logic import LogicParser
from nltk.inference import Prover9
import regex as re  
from typing import Optional
import os
import gc
import copy
from collections import Counter
from torch.utils.data import DataLoader
from tqdm import tqdm

In [4]:
import warnings

# Chặn toàn bộ UserWarning phát ra từ module save_and_load của thư viện peft
warnings.filterwarnings(
    "ignore", 
    category=UserWarning, 
    module="peft.utils.save_and_load"
)

In [5]:
class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        item["fol_premises"] = item.get("fol_premises", None)
        item["fol_conclusion"] = item.get("fol_conclusion", None)
        return {
            "id": item["id"],
            "nat_premises": item["nat_premises"],
            "fol_premises":  "" if item["fol_premises"] is None else item["fol_premises"],
            "nat_conclusion": item["nat_conclusion"],
            "fol_conclusion": "" if item["fol_conclusion"] is None else item["fol_conclusion"],
            "label": item["label"]
        }


class FOLModel:
    def __init__(
        self,
        model_name: str = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        output_dir: str = "/kaggle/working/fol_model",
        adapter_dir: str = "/kaggle/input/models/ductri0981/fol-model/transformers/default/2",
        max_length: int = 768,
        local_files_only=True
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        self.adapter_dir = adapter_dir
        
        #Load tokenizer
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            local_files_only=local_files_only
        )

        #Load base model.
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.config.use_cache = False
        self.tokenizer.padding_side = "left"
        self._load_or_init_lora()
        
    def _init_lora(self):
        print("Initializing LoRA...")
    
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
    
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()

    def _load_or_init_lora(self):
        if isinstance(self.model, PeftModel):
            print("Resetting to base model...")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                dtype=torch.float16,
                device_map={"": 0},
                trust_remote_code=True,
                local_files_only=True
            )
        
        if os.path.exists(self.adapter_dir):
            print("Loading existing LoRA adapter...")

            self.model = PeftModel.from_pretrained(
                self.model,
                self.adapter_dir,
                is_trainable=True,
                local_files_only=True
            )
        else:
            print("No adapter found → initializing new LoRA")
            self._init_lora()

    def _prompt_template(self, data_example, is_predict=False):
        premises_list = [
            s.strip() for s in data_example['nat_premises'].split('.') if s.strip()
        ]
        formatted_premises = "\n".join(
            [f"{i+1}. {s}." for i, s in enumerate(premises_list)]
        )
        formatted_all = formatted_premises + f"\n{len(premises_list)+1}. {data_example['nat_conclusion']}"
        output_part = "" if is_predict else f"{data_example['fol_premises']}\n{data_example['fol_conclusion']}"
        prompt = f"""### Role
You are a precise Natural Language to First-Order Logic (FOL) translator. Your task is to perform a direct, line-by-line mapping of sentences into formal logic.

### Logical Symbols
Strictly use these symbols:
- Universal: ∀
- Existential: ∃
- Conjunction: ∧
- Disjunction: ∨
- Exclusive Or: ⊕
- Negation: ¬
- Implication: →
- Equivalence: ↔

### Strict Constraints
1. **One-to-One Parity**: The number of FOL expressions in the output must exactly match the number of sentences in the Natural Language input. 
2. **Line Separation**: Each FOL expression must be placed on a new line. Do not use any other delimiters or numbering.
3. **Predicate Consistency**: 
   - Use a single, consistent name for each property or relation. 
   - Do not use synonyms (e.g., if you use "Student(x)", do not use "is_student(x)" or "IsStudent(x)" elsewhere).
   - Prefer concise noun-based or verb-based naming (e.g., "Teacher(x)", "Loves(x, y)").
4. **No Metadata**: Output ONLY the FOL expressions. No explanations, no thought process, and no introductory remarks.

Now apply the rules strictly to the following input:

### Input:
{formatted_all}

### Output:
{output_part}
"""
        return {"text": prompt}
        
    def _tokenize(self, data):
        prompt = data['text']
        tokenized = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )
        labels = tokenized["input_ids"].copy()
        # Only predict next token after "output prompt"
        output_start = prompt.index("### Output:")
        output_tokens = self.tokenizer(prompt[:output_start])["input_ids"]
        #Label from start to output tokens => mask with -100.
        labels[:len(output_tokens)] = [-100] * len(output_tokens)
        tokenized["labels"] = labels
        return tokenized

    def prepare_sample(self, data_example):
        """Tiền xử lý 1 mẫu dữ liệu hoàn chỉnh thành các Token ID trước khi đưa vào finetune_set"""
        prompt_dict = self._prompt_template(data_example)
        tokenized_dict = self._tokenize(prompt_dict)
        
        # Chỉ trả về các trường cần thiết để huấn luyện, tiết kiệm bộ nhớ RAM
        return {
            "input_ids": tokenized_dict["input_ids"],
            "attention_mask": tokenized_dict["attention_mask"],
            "labels": tokenized_dict["labels"]
        }
    in
    def train(
        self,
        train_data: ManualDataset,
        epochs: int = 1,
        batch_size: int = 4,
        learning_rate: float = 1e-4,
        gradient_accumulation_steps=1
    ):
        
        print("Loading finetuning data...")
        train_data = Dataset.from_list(train_data.data)
        train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

        training_args = TrainingArguments(
            output_dir=self.output_dir,
            
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            num_train_epochs=epochs,
            learning_rate=learning_rate,
            
            logging_steps=10,
            report_to="none",

            bf16=True,
            tf32=True,
            dataloader_num_workers=4,

            save_total_limit=1
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_data,
        )
        
        trainer.train()
        
        del trainer
        gc.collect()
        torch.cuda.empty_cache()
        
    def predict(self, dataset: ManualDataset, max_new_tokens: int = 768, num_outputs: int = 10):
        loader = DataLoader(
            dataset,
            batch_size=32,
            shuffle=False,
        )
        self.model.eval()
        data = []
    
        with torch.no_grad():
            loop = tqdm(loader)
    
            for batch in loop:
    
                ids = batch["id"]
                premises = batch["nat_premises"]
                conclusions = batch["nat_conclusion"]
                labels = batch["label"]
    
                # ensure list
                if isinstance(premises, str):
                    premises = [premises]
                    conclusions = [conclusions]
                    ids = [ids]
                    labels = [labels]
    
                batch_size = len(premises)
    
                # 1. build prompts correctly
                prompts = []
                for i in range(batch_size):
    
                    sample = {
                        "nat_premises": premises[i],
                        "nat_conclusion": conclusions[i]
                    }
    
                    prompts.append(self._prompt_template(sample, is_predict=True)["text"])
    
                # 2. tokenize batch
                inputs = self.tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=self.max_length
                ).to(self.model.device)
    
                # 3. generate
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=1.0,
                    min_p=0.05,
                    num_return_sequences=num_outputs,
                    pad_token_id=self.tokenizer.eos_token_id
                )
    
                # 4. decode
                decoded_outputs = self.tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )
    
                # 5. reshape results
                for i in range(batch_size):
    
                    sample_outputs = []
    
                    for j in range(num_outputs):
    
                        idx = i * num_outputs + j
                        text = decoded_outputs[idx]
    
                        if "### Output:" in text:
                            text = text.split("### Output:")[-1].strip()
                        else:
                            text = text[len(prompts[i]):].strip()
    
                        sample_outputs.append(text)
    
                    data.append({
                        "id": ids[i],
                        "natural": premises[i] + "\n" + conclusions[i],
                        "predictions": [
                            {"fol": fol, "label": None}
                            for fol in sample_outputs
                        ],
                        "label": labels[i]
                    })
    
        return data

In [6]:
class Engine:
    def __init__(self, prover9_path='C:\\Program Files (x86)\\Prover9-Mace4\\bin-win32'):
        self.parser = LogicParser()
        self.prover9_path = prover9_path
        self.prover = Prover9()
        self.prover.config_prover9(prover9_path)

    def _transform_xor(self, text):
        """
        Transform A ⊕ B to ¬(A ⟷ B)
        """
        while '⊕' in text:
            idx = text.find('⊕')
            
            # Find left component
            lhs_end = idx
            while lhs_end > 0 and text[lhs_end-1].isspace():
                lhs_end -= 1
                
            curr = lhs_end - 1
    
            if text[curr] == ')':
                balance = 0
                while curr >= 0:
                    if text[curr] == ')': balance += 1
                    elif text[curr] == '(': balance -= 1
                    curr -= 1
                    if balance == 0: break
                while curr >= 0 and (text[curr].isalnum() or text[curr] == '_'):
                    curr -= 1
                lhs_start = curr + 1
            else:
                while curr >= 0 and (text[curr].isalnum() or text[curr] == '_'):
                    curr -= 1
                lhs_start = curr + 1
                
            lhs = text[lhs_start:lhs_end]
    
            # Find right component
            rhs_start = idx + 1
            while rhs_start < len(text) and text[rhs_start].isspace():
                rhs_start += 1
                
            curr = rhs_start
            if text[curr].isalnum() or text[curr] == '_':
                while curr < len(text) and (text[curr].isalnum() or text[curr] == '_'):
                    curr += 1
                if curr < len(text) and text[curr] == '(':
                    balance = 0
                    while curr < len(text):
                        if text[curr] == '(': balance += 1
                        elif text[curr] == ')': balance -= 1
                        curr += 1
                        if balance == 0: break
                rhs_end = curr
            elif text[curr] == '(':
                balance = 0
                while curr < len(text):
                    if text[curr] == '(': balance += 1
                    elif text[curr] == ')': balance -= 1
                    curr += 1
                    if balance == 0: break
                rhs_end = curr
            else:
                rhs_end = curr + 1
                
            rhs = text[rhs_start:rhs_end]
    
            # Transform XOR
            new_expression = f"¬({lhs} ⟷ {rhs})"
            text = text[:lhs_start] + new_expression + text[rhs_end:]
    
        return text
    
    def _translate_fol(self, fol_text: str):
        # Transform XOR
        fol_text = self._transform_xor(fol_text)
        
        # '-' --> '_'
        fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

        replacements = {
            '∀': 'all ', 
            '∃': 'exists ',
            '∧': '&', 
            '∨': '|',
            '⊕': '^',
            '¬': '-',
            '→': '->', 
            '⟷': '<->',
            '↔': '<->'
        }
        for k, v in replacements.items():
            fol_text = fol_text.replace(k, v)

        # Add dot: "all x (P(x))" --> "all x. (P(x))"
        fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

        # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
        words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
        reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
        
        for w in set(words):
            if w not in reserved_words:
                fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
        return fol_text


    def _is_valid_fol(self, fol_list):
        try:
            for line in fol_list:
                if line.strip():
                    self.parser.parse(line)
            return True
        except Exception:
            return False

    def _check_conclusion(self, premises, conclusion):
        '''
        This function is the engine. It checks whether the conclusion is True/False/Uncertain based on the list of premises.
        Args:
            premises: list[str]
            conclusion: str
        Returns: 
            str ("True"/"False"/"Uncertain")
        '''
        # Read fol strings
        translated_premises = [self._translate_fol(p) for p in premises]
        translated_conclusion = self._translate_fol(conclusion)

        if (not self._is_valid_fol(translated_premises) or not self._is_valid_fol([translated_conclusion])):
            error_msg = f"Invalid FOL syntax detected!"
            raise ValueError(error_msg)
        
        try:
            parsed_premises = [self.parser.parse(p) for p in translated_premises]
            parsed_conclusion = self.parser.parse(translated_conclusion)
        except Exception as e:
            raise f"Error: {e}"

        # Check conclusion
        if self.prover.prove(parsed_conclusion, []):
            raise Exception("Error: The conclusion itself is True!")
        elif self.prover.prove(parsed_conclusion.negate(), []):
            raise Exception("Error: The conclusion itself is False!")
        elif not self.prover.prove(None, parsed_premises):
            is_true = self.prover.prove(parsed_conclusion, parsed_premises)
            if is_true:
                return "True"

            is_false = self.prover.prove(parsed_conclusion.negate(), parsed_premises)
            if is_false:
                return "False"

            return "Uncertain"
        else: 
            raise Exception("Error: The premises are inconsistent!")
    
    def check_conclusion(self, data):
        fol_list = data["predictions"]
        new_fol_list = []
        for fol in fol_list:
            try:
                premises = fol["fol"].split("\n")[:-1]
                conclusion = fol["fol"].split("\n")[-1]
                result = self._check_conclusion(premises, conclusion)

                if result == "True":
                    fol["label"] = "True"
                elif result == "False":   # FIX 2
                    fol["label"] = "False"
                else:
                    fol["label"] = "Uncertain"
                new_fol_list.append(fol)
            except Exception as e:
                # print(e)
                continue

        data["predictions"] = new_fol_list
        return data

In [7]:
def lv_distance(s1, s2):
    if len(s1) < len(s2):
        return lv_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

class DataFilter:
    def __init__(self):
        self.duplicate_count = 0
        self.syntax_error_count = 0
        self.length_ratio_count = 0
    
    def _is_valid_syntax(self, fol_str: str) -> bool:
        if not fol_str:
            return False
            
        stack = []
        matching_bracket = {')': '(', ']': '[', '}': '{'}
        
        for char in fol_str:                   
            if char in matching_bracket.values():
                stack.append(char)
            elif char in matching_bracket.keys():
                if not stack or stack.pop() != matching_bracket[char]:
                    return False

        return len(stack) == 0
    
    
    def _is_valid_length_ratio(self, nat_str: str, fol_str: str, lowerbound_ratio: float=0.5, upperbound_ratio: float=2.0) -> bool:
        nat_length = len(nat_str)
        if nat_length == 0:
            return False
    
        fol_length = len(fol_str)
        ratio = fol_length / nat_length
        
        return lowerbound_ratio <= ratio <= upperbound_ratio


    def _update_names(self, fol, ratio_threshold=1/3):
        pattern = r'(\w+)\('
        matches = re.findall(pattern, fol)
    
        counts = Counter(matches)
        all_names = list(set(matches))
    
        single_names = [name for name, count in counts.items() if count == 1]
    
        update_names = {}
    
        for n1 in single_names:
            for n2 in all_names:
                if n2 in single_names and len(n2) > len(n1):
                    continue
                if n1 != n2 and lv_distance(n1, n2) <= ratio_threshold * max(len(n1), len(n2)):
                    update_names[n1] = n2
    
        return update_names
    
    
    def filter(self, entry: dict, lowerbound_ratio:float=0.5, upperbound_ratio:float=2.0) -> dict:
        valid_predictions = []
        nat_str = entry.get("natural", "")
        
        natural_text = entry.get("natural", "").strip()
        sentences = [s for s in natural_text.split('.') if s.strip()]
        nat_length = len(sentences)

        for pred in entry.get("predictions", []):
            fol = pred.get("fol", "")
            
            # Validate syntax (bracket balancing)
            if not self._is_valid_syntax(fol):
                self.syntax_error_count += 1
                continue
            
            premise_list = [p.strip() for p in fol.split('\n') if p.strip()]
            conclusion = premise_list[-1]
            
            premise_list = list(dict.fromkeys(premise_list[:-1]))

            # if len(premise_list) != (nat_length - 1):
            #     self.duplicate_count += 1
            #     continue
            
            # Check for length ratio
            if not self._is_valid_length_ratio(nat_str, fol, lowerbound_ratio, upperbound_ratio):
                self.length_ratio_count += 1
                continue

            valid_fol = "\n".join(premise_list + [conclusion])
            update = self._update_names(valid_fol)
            for key, value in update.items():
                pattern = fr'(?<![a-zA-Z0-9]){key}\('
                valid_fol = re.sub(pattern, value + "(", valid_fol)

            valid_predictions.append({'fol': valid_fol, 'label': None})
            
        filtered_element = entry.copy()
        filtered_element["predictions"] = valid_predictions

        return filtered_element

def filter2_pred(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None

    count = {"True": 0, "False": 0, "Uncertain": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    if count[ground_truth] / len(fol_predictions) > threshold:
        return True
    return False

def filter2(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None

    count = {"True": 0, "False": 0, "Uncertain": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    if count[ground_truth] / len(fol_predictions) > threshold:
        positive_sample = [prediction for prediction in fol_predictions if prediction["label"] == ground_truth]
        negative_sample = [prediction for prediction in fol_predictions if prediction["label"] != ground_truth]
        return positive_sample, negative_sample

    return None

In [8]:
!cp -rf /kaggle/input/models/ductri0981/fol-model/transformers/default/2 /kaggle/working/lora_adapter

In [9]:
class Pipeline:
    def __init__(self, fol_model):
        self.fol_model = fol_model
        self.filter_1 = DataFilter()
        self.engine = Engine(prover9_path="/kaggle/working/Prover9-LADR-2026-4F/bin")
    
    def get_data(self, file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            dataset = json.load(f)
        return dataset

    def __save_file_prediction(self, predictions, file_name="predictions.json"):
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(predictions, f, indent=4, ensure_ascii=False)
            
    def train(self, train_set_path,
              valid_set_path,
              silver_set_path,
              new_train_set_path,
              folio_set_path,
              batch_pop=32,
              max_steps=3, batchsize=4, previous_step=1):

        best_accuracy = 0.0

        train_set = self.get_data(train_set_path)
        valid_set = self.get_data(valid_set_path)
        
        finetune_set = []
        silver_set = []

        folio_set = self.get_data(folio_set_path)
        for i in range(len(folio_set)):
            processed_tensor_sample = self.fol_model.prepare_sample(folio_set[i])
            finetune_set.append({
                "item": processed_tensor_sample,
                "usage_count": 0
            })
            silver_set.append(folio_set[i])

        print(f"Bắt đầu quy trình tự học với tổng {len(train_set)} mẫu trong dataset gốc.")

        for step in range(max_steps):
            print(f"\n========== STEP {step + 1 + previous_step}/{max_steps + previous_step} ==========")

            if len(train_set) == 0:
                print("Đã lấy hết dữ liệu từ train_set ban đầu. Tiếp theo chỉ thực hiện finetune.")
            else:
                # Lấy batchsize mẫu từ trên xuống và xóa khỏi train_set
                current_batch_size = min(batch_pop, len(train_set))
                batch_data = train_set[:current_batch_size]
                train_set = train_set[current_batch_size:]
    
                # Mô hình dự đoán
                batch_data = ManualDataset(batch_data)
                predicted_batch = self.fol_model.predict(batch_data)
                # predicted_batch = self.format_predict_dataset(batch_data)
    
                for original_data, pred_data in zip(batch_data, predicted_batch):
                    is_success = False
                    
                    pred_data = self.filter_1.filter(pred_data)
                    pred_data = self.engine.check_conclusion(pred_data)
                    
                    result_filter2 = filter2(pred_data['predictions'], pred_data["label"])
                    if result_filter2 is not None:
                        positive_fol, _ = result_filter2
                        chosen_pred = positive_fol[0]
    
                        nat_prem, nat_conc = pred_data["natural"].split("\n")
    
                        formatted_sample = {
                            "id": pred_data["id"],
                            "nat_premises": nat_prem,
                            "nat_conclusion": nat_conc,
                            "fol_premises": '\n'.join(chosen_pred['fol'].split('\n')[:-1]),
                            "fol_conclusion": chosen_pred['fol'].split('\n')[-1],
                            "label": pred_data["label"]
                        }
    
                        silver_set.append(formatted_sample)
    
                        processed_tensor_sample = self.fol_model.prepare_sample(formatted_sample)
                        finetune_set.append({
                            "item": processed_tensor_sample, # Đã là input_ids, attention_mask, labels
                            "usage_count": 0
                        })
                        
                        is_success = True
    
                    if not is_success:
                        train_set.append(original_data)
    
                print(f"Kích thước finetune_set hiện tại: {len(finetune_set)}")
    
                # Huấn luyện bằng dữ liệu từ finetune_set
                if len(finetune_set) >= batch_pop:
                    weights = np.array([1.0 / (sample["usage_count"] + 1) for sample in finetune_set])
                    probs = weights / weights.sum() # Chuẩn hóa để tổng xác suất = 1.0
    
                    sampled_indices = np.random.choice(
                        len(finetune_set), 
                        size=batch_pop, 
                        replace=False, 
                        p=probs
                    )
    
                    train_batch = []
                    for idx in sampled_indices:
                        finetune_set[idx]["usage_count"] += 1
                        train_batch.append(finetune_set[idx]["item"])
    
                    # print(f"Bắt đầu huấn luyện mô hình với {batchsize} mẫu từ finetune_set...")
                    train_batch = ManualDataset(train_batch)
                    self.fol_model.train(
                        train_data=train_batch,
                        batch_size=batchsize
                    )
                else:
                    print("Chưa đủ dữ liệu trong finetune_set để tạo batch huấn luyện. Chuyển sang step tiếp theo...")

            # 50 steps valid một lần, cũng như lưu silver_data và new_train_set một lần
            if step % 50 == 0:
                if len(silver_set) > 0:
                    with open(silver_set_path, "w", encoding="utf-8") as f:
                        json.dump(silver_set, f, ensure_ascii=False, indent=4)
                    print(f"Đã lưu/cập nhật {len(silver_set)} mẫu vào {silver_set_path}")

                with open(new_train_set_path, "w", encoding="utf-8") as f:
                    json.dump(train_set, f, ensure_ascii=False, indent=4)
                print(f"Đã cập nhật new_train_set vào {new_train_set_path}")
                
                # Valid thủ công ở đây
                count_correct = 0
                valid_set = ManualDataset(valid_set)
                predictions = self.fol_model.predict(valid_set, num_outputs=10)
                # predictions = self.format_predict_dataset(valid_set, num_outputs=10)
                # self.__save_file_prediction(predictions)
                filtered_predictions = []
                for prediction in predictions:
                    prediction = self.filter_1.filter(prediction)
                    prediction = self.engine.check_conclusion(prediction)
                    filtered_predictions.append(prediction)
                    count_correct += (filter2(prediction['predictions'], prediction["label"]) is not None)
                self.__save_file_prediction(filtered_predictions)
                accuracy = count_correct / len(predictions)
                print("Validation accuracy:", accuracy)
                if accuracy > best_accuracy or accuracy == 1:
                    self.fol_model.model.save_pretrained(self.fol_model.adapter_dir)
                    self.fol_model.tokenizer.save_pretrained(self.fol_model.adapter_dir)
                    best_accuracy = accuracy

In [10]:
fol_model = FOLModel(
    model_name="/kaggle/input/models/ductri0981/deepseek-model/transformers/default/3/deepseek_model",
    adapter_dir="/kaggle/working/lora_adapter",
    local_files_only=True
)

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading existing LoRA adapter...


In [ ]:
pipeline = Pipeline(fol_model=fol_model)
pipeline.train(
    train_set_path="/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/train_without_fol.json",
    valid_set_path="/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/valid_without_fol.json",
    silver_set_path="/kaggle/working/silver_data.json",
    new_train_set_path="/kaggle/working/new_train_set.json",
    folio_set_path="/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/folio_train.json",
    batch_pop=32,
    max_steps=201,
    batchsize=8,
    previous_step=0
)


Bắt đầu quy trình tự học với tổng 4199 mẫu trong dataset gốc.

========== STEP 1/201 ==========


100%|██████████| 1/1 [02:52<00:00, 172.05s/it]


Kích thước finetune_set hiện tại: 873
Loading finetuning data...


Step,Training Loss


Đã lưu/cập nhật 873 mẫu vào /kaggle/working/silver_data.json
Đã cập nhật new_train_set vào /kaggle/working/new_train_set.json


...

Kích thước finetune_set hiện tại: 1557
Loading finetuning data...


Step,Training Loss



========== STEP 53/201 ==========


  0%|          | 0/1 [01:08<?, ?it/s]


KeyboardInterrupt: 